# Cohort A: link cancer registry diagnoses to EHR diagnoses (Cartesian/crosswalk setup)

This notebook builds the reference views used to connect two independent sources of cancer
diagnoses for **Cohort A**:

1. The **CIPOC cancer registry** (`linkage_files.cipoc_xwalk`) — the ground-truth source, limited
   here to diagnoses on/after 2015-10-01 (the ICD-10 transition date).
2. The **EHR condition data** (OMOP `condition_occurrence`), which uses ICD-10-CM codes mapped
   through OMOP concept relationships.

Both sources use different coding systems for "type of cancer," so the first few views build a
crosswalk (`WHOCartesian`) between WHO site/histology codes and a common `Rollup2` category, then
match registry patients to EHR patients and EHR diagnosis codes to that same category. The output,
`cohort_a_derived.cohort_a_source`, is a per-patient, per-cancer-episode table flagging whether the
episode was seen in the registry, the EHR, or both (`match_type`).

Downstream notebook: `04_set_up_cohort_a_feature_table` builds features on top of this table.

In [2]:
USE CATALOG your_catalog;

SyntaxError: invalid syntax (2480928310.py, line 1)

### Reference crosswalk: WHO site/histology codes
`WHOCartesian` cross-joins the WHO cancer site code reference table with the WHO histology code
reference table (matched on `Category`), producing the full site/histology combinations later used
to bucket ICD codes into `Rollup2` cancer categories.

In [ ]:
%sql 
drop view if exists WHOCartesian;
CREATE view WHOCartesian as
SELECT sc.*, hc.SiteRange, hc.HistologyExplode
FROM reference.who_site_codes sc LEFT JOIN reference.who_histology_codes hc ON sc.Category = hc.Category

### Registry patients (Cohort A)
Pulls CIPOC registry diagnoses for patients also present in `cohort_a.person`, matched via the
`cipoc_xwalk` crosswalk table, restricted to diagnoses on/after the ICD-10 transition date.

In [ ]:
%sql
drop view if exists registry_pts;
create view registry_pts as 
--grab the crosswalk, limit to post-ICD-10 time range
select p.person_id, cx.cancer_site as registry_site, cx.date_of_diagnosis, cx.primary_site as registry_dx_code, cx.sequence_number_central   
from linkage_files.cipoc_xwalk cx JOIN cohort_a.person p ON cx.pat_id = p.person_source_value
where cx.date_of_diagnosis >= '2015-10-01'

### EHR condition codes present in the data, mapped back to ICD-10-CM
The EHR stores `condition_concept_id`s (OMOP standard concepts). This view walks the OMOP
`concept_relationship` ("Maps to") to recover the original ICD-10-CM "C" codes (malignant
neoplasms) that map to each concept actually observed in `cohort_a.condition_occurrence`.

In [ ]:
%sql
--convert condition_concept_ids that are actually in the data back to ICD-10s
drop view if exists all_c_codes_in_data;
create view all_c_codes_in_data as 
SELECT distinct c.concept_id as original_concept_id, c2.*
FROM cohort_a.concept c JOIN cohort_a.condition_occurrence co ON c.concept_id = co.condition_concept_id 
    JOIN cohort_a.concept_relationship cr ON c.concept_id = cr.concept_id_2 and relationship_id = 'Maps to'
    JOIN cohort_a.concept c2 ON c2.concept_id = cr.concept_id_1 and c2.vocabulary_id = 'ICD10CM' and c2.concept_code LIKE 'C%' and c2.invalid_reason is null

### Classify each EHR cancer diagnosis into a Rollup2 category
Combines the WHO crosswalk (`WHOCartesian`) with an additional EHR-specific ICD mapping table
(`reference.who_ehr_mapping`) into one master code list, then joins each EHR diagnosis's truncated
ICD-10 code to that list to assign a `Rollup2` cancer category (falling back to `'Unmapped'`).

In [ ]:
%sql
drop view if exists cancer_code_density;
create view cancer_code_density as 
--compile EHR cancers
with master_code_list as (
    select cast(ExplodedSiteRange as string), Rollup, Rollup2 from WHOCartesian
    UNION
    select replace(replace(ICDCode,'.',''),'C','') as ExplodedSiteRange, Rollup, Rollup2 from reference.who_ehr_mapping
),

prelim_cancers as (
SELECT distinct co.person_id as ehr_person_id, co.condition_occurrence_id, co.condition_concept_id,  
case when length(left(replace(a.concept_code,'.',''),4)) = 3 then left(replace(a.concept_code,'.',''),4) || '0'  else left(replace(a.concept_code,'.',''),4) end as truncated_stripped_icd_ehr, co.condition_start_date
FROM cohort_a.condition_occurrence co 
JOIN all_c_codes_in_data a ON co.condition_concept_id = a.original_concept_id
)

select p.*, 
case when m_ehr.Rollup2 is null then 'Unmapped' else m_ehr.Rollup2 end as Rollup2 
from prelim_cancers p LEFT JOIN master_code_list m_ehr ON replace(replace(truncated_stripped_icd_ehr,'.',''),'C','') = m_ehr.ExplodedSiteRange

### Collapse diagnosis codes into EHR "episodes"
For each patient + cancer category, collapses all matching diagnosis codes into a single episode
spanning the first to last diagnosis date (`episode_start` / `episode_end`).

In [ ]:
%sql
drop view if exists ehr_episode_dates;
create view ehr_episode_dates as 
--create cancer episodes in the EHR data
SELECT ehr_person_id, Rollup2, min(condition_start_date) as episode_start, max(condition_start_date) as episode_end
FROM cancer_code_density
group by ehr_person_id, Rollup2

### Match EHR episodes to registry diagnoses
Core linkage step, unioning three groups:
- **Registry and EHR**: patient/cancer-category pairs found in both sources (within the EHR episode
  window), with `daysbt` = days between the registry diagnosis date and the EHR episode start.
- **Registry only, no EHR**: registry diagnoses with no matching EHR episode 
- **EHR only, no registry**: EHR episodes with no matching registry diagnosis 

In [ ]:
%sql
drop view if exists episode_matching;
create view episode_matching as
--see which cancers are coming from which source
with in_ehr_and_reg as (
--patients in both EHR and registry  
SELECT distinct co.ehr_person_id, n.person_id as registry_person_id, co.condition_occurrence_id as ehr_cancer_id, co.condition_concept_id as original_concept_id_ehr, truncated_stripped_icd_ehr, e.episode_start as ehr_episode_start, e.episode_end as ehr_episode_end, 'Registry and EHR' as match_type, e.Rollup2 as ehr_rollup2, n.date_of_diagnosis as registry_cancer_dx_date, datediff(n.date_of_diagnosis, e.episode_start) as daysbt, n.person_id || n.sequence_number_central as rownum, n.registry_site 
FROM cancer_code_density co
JOIN ehr_episode_dates e ON co.ehr_person_id = e.ehr_person_id and co.Rollup2 = e.Rollup2
JOIN registry_pts n ON co.ehr_person_id = n.person_id
WHERE e.episode_start >= '2015-10-01'), 

registry_only as (
--cancers in the registry that have no EHR match with an EHR episode
--possible reasons: MRN merges, patient in UNC records for reasons other than cancer
SELECT distinct co.ehr_person_id, n.person_id as registry_person_id, co.condition_occurrence_id as ehr_cancer_id, co.condition_concept_id as original_concept_id_ehr, truncated_stripped_icd_ehr, e.episode_start as ehr_episode_start, e.episode_end as ehr_episode_end, 'Registry only no EHR' as match_type, e.Rollup2 as ehr_rollup2, n.date_of_diagnosis as registry_cancer_dx_date, datediff(n.date_of_diagnosis, e.episode_start) as daysbt, n.person_id || n.sequence_number_central as rownum, n.registry_site
FROM cancer_code_density co
JOIN ehr_episode_dates e ON co.ehr_person_id = e.ehr_person_id and co.Rollup2 = e.Rollup2
RIGHT JOIN registry_pts n ON co.ehr_person_id = n.person_id  
WHERE co.ehr_person_id is null and n.person_id || n.sequence_number_central NOT IN (select distinct rownum from in_ehr_and_reg)
),

ehr_only as (
--cancers in the EHR that have no registry match 
--possible reasons: MRN merges, registry cancer is pre-2015 but still mentioned in EHR
SELECT distinct co.ehr_person_id, n.person_id as registry_person_id, co.condition_occurrence_id as ehr_cancer_id, co.condition_concept_id as original_concept_id_ehr, truncated_stripped_icd_ehr, e.episode_start as ehr_episode_start, e.episode_end as ehr_episode_end, 'EHR only no registry' as match_type, e.Rollup2 as ehr_rollup2, n.date_of_diagnosis as registry_cancer_dx_date, datediff(n.date_of_diagnosis, e.episode_start) as daysbt, n.person_id || n.sequence_number_central as rownum, n.registry_site
FROM cancer_code_density co
JOIN ehr_episode_dates e ON co.ehr_person_id = e.ehr_person_id and co.Rollup2 = e.Rollup2
LEFT JOIN registry_pts n ON co.ehr_person_id = n.person_id 
WHERE n.person_id is null
and e.episode_start >= '2015-10-01')

select * from in_ehr_and_reg
UNION
select * from registry_only
UNION
select * from ehr_only

### Persist the final Cohort A source table
Simplifies `episode_matching` down to the columns needed downstream and saves it as
`cohort_a_derived.cohort_a_source`.

In [ ]:
%sql
--simplifying the output from the view above
drop table if exists cohort_a_derived.cohort_a_source;
create table cohort_a_derived.cohort_a_source as
SELECT distinct e.ehr_person_id, e.registry_person_id, e.ehr_episode_start, e.ehr_episode_end, e.match_type, e.ehr_rollup2, e.registry_cancer_dx_date, e.daysbt, e.rownum as registry_rownum, e.registry_site
FROM episode_matching e 